# A(Dr) — retrain the original MLP on the retain set (GPU)

Reference ceiling for unlearning: the same architecture `[85,111]`, trained **only on Dr**.

**Before running:** `Runtime -> Change runtime type -> Hardware accelerator = GPU`.

In [ ]:
# 1) get the code + data (data is committed in the repo)
!git clone -q https://github.com/julpfi/sapienza_tc_tim_machine_unlearning.git
%cd sapienza_tc_tim_machine_unlearning

In [ ]:
# 2) imports + GPU check
import glob, time
import numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from pathlib import Path
from utils import functions as uf
from utils import eval as ue
from utils.model import DynamicMLP

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')
torch.manual_seed(42); np.random.seed(42)

In [ ]:
# 3) load data, split forget/retain, then 85/15 train/val (retain only)
df = pd.concat((pd.read_csv(f, sep=';') for f in glob.glob('./data/*c000.csv')), ignore_index=True)
fids = pd.read_csv('data/forget_data.csv')['user_id']
retain_df = df[~df['user_id'].isin(fids)].reset_index(drop=True)   # Dr only (forget excluded)
train_df, val_df = train_test_split(retain_df, test_size=0.15, random_state=42)
print('retain:', len(retain_df), '| train:', len(train_df), '| val:', len(val_df))

In [ ]:
# 4) features (imputer fit on train; NO scaling -> same recipe as the given model)
Xtr, ytr, feat, targ = uf.prepare_data(train_df, id_col='user_id', target_prefix='target__')
Xva, yva, _, _       = uf.prepare_data(val_df,   id_col='user_id', target_prefix='target__')
imp = SimpleImputer(strategy='median')
Xtr = imp.fit_transform(Xtr).astype(np.float32)
Xva = imp.transform(Xva).astype(np.float32)

pc = ytr.sum(0)
pos_weights = torch.tensor((len(ytr) - pc) / (pc + 1e-6), device=device).clamp(0.1, 100.0).float()
print('X:', Xtr.shape, '| positives/user (train):', ytr.sum(1).mean().round(3))

In [ ]:
# 5) architecture + hyperparameters from the ORIGINAL artifact (same model, trained from scratch)
payload = uf.load_pickle(Path('data') / 'model_artifact')
arch = payload['architecture']; best = payload['best_hyperparameters']
print('architecture:', arch)
print('best_hyperparameters:', best)

EPOCHS = int(best.get('epochs', 55))
LR     = float(best.get('lr', 2e-3))
BATCH  = 512

In [ ]:
# 6) train A(Dr) on GPU, track val Precision@10
loader = DataLoader(
    TensorDataset(torch.tensor(Xtr), torch.tensor(ytr, dtype=torch.float32)),
    batch_size=BATCH, shuffle=True)

model = DynamicMLP(arch['input_dim'], arch['hidden_layers'], arch['num_outputs']).to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)
crit = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

best_p10 = 0.0
t0 = time.perf_counter()
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
    if (epoch + 1) % 5 == 0 or epoch == EPOCHS - 1:
        p10 = ue.precision_at_k(model, Xva, yva)   # val = held-out retain
        best_p10 = max(best_p10, p10)
        print(f'epoch {epoch+1:>3}/{EPOCHS}  val P@10 = {p10:.4f}')
print(f'\nA(Dr) best val P@10 = {best_p10:.4f}   (time {time.perf_counter()-t0:.1f}s)')

## Interpretazione

- `A(Dr)` = tetto di **riferimento dell'unlearning** per l'architettura `[85,111]`.
- Se esce **~0.64**, conferma: SSD (0.636) sta gia' raggiungendo il gold standard, e 0.64 e' il limite di **questa architettura** (non del metodo).
- Vuoi il tetto **vero** dei dati? Cambia `arch['hidden_layers']` in `[256,128]` o `[512,256]` nella cella 6 e rilancia: vedi fin dove sale la precision con piu' capacita'.